In [90]:
import os
import streamlit as st
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [91]:
#loading api key from .env file
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY IS NOT PRESENT IN MY .ENV FILE.")

In [92]:
#PDF TO TEXT
def read_pdfs(pdf_fileS):
    all_texts=""
    for pdf in pdf_fileS:
        reader = PdfReader(pdf)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                all_text+=text
    return all_text

In [93]:
def read_pdfs(pdf_files):
    all_text = ""

    for pdf in pdf_files:
        reader = PdfReader(pdf)

        for page in reader.pages:
            text = page.extract_text()
            if text:
                all_text += text

    return all_text

In [94]:
text=read_pdfs(['attention.pdf'])

In [96]:
print(text)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model a

In [97]:
#chunks-trying to make pieces small enough for the model preserving the context of the text
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
    )
    return splitter.split_text(text)

In [98]:
chunks=split_text(text)

In [99]:
len(chunks)

132

In [100]:
chunks[0]

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto'

In [101]:
# enbedding of chunks
def create_embeddings():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [102]:
# build my vector database to store the embedding
def build_vector_store(chunks):
    embedding = create_embeddings()
    documents=[Document(page_content=chunk) for chunk in chunks]
    vector_store = FAISS.from_documents(documents,embedding)
    vector_store.save_local("faiss_index")

In [103]:
build_vector_store(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4663.00it/s]


In [104]:
def load_vector_store():
    embedding = create_embeddings()
    return FAISS.load_local("faiss_index", embedding, allow_dangerous_deserialization=True)

In [105]:
#RETRIEVING THE CHUNKS/EMBEDDINGS
def retrieve_chunks(question):
    vector_store = load_vector_store()
    return vector_store.similarity_search(question, k=10)

In [106]:
docs=retrieve_chunks("explain self attention in transformer")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2099.73it/s]


In [107]:
for d in docs:
    print(d.page_content)

dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
language modeling tasks [28].
To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
multi-headed self-attention.
For translation tasks, the Transformer can be trained signiﬁcantly faster than architectures based
on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014
results to the base model.
7 Conclusion
In this work, we presented the Transformer, the ﬁrst sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with
multi-headed self-attention.
[9], consuming t

In [108]:
def get_prompt_and_llm():
    prompt_template="""
You are an AI assistant.
Answer the question using only the context below.
If the answer is not present in the context, say exactly:
"THE ANSWER IS NOT AVAILABLE IN THE PROVIDED CONTEXT."
The answer has to be in bullet point wise, each point not exceeding 200 words.
THE answer has to be in  away that a CLASS 10TH GRADE STUDENT UNDERSTANDS IT.

Context:
{context}
Question:
{question}

Answer:
"""
    prompt=PromptTemplate(template=prompt_template,input_variables=['context','question'])
    llm= ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite',temperature=1.9)
    return prompt,llm

In [109]:
def answer_question(question):
    docs=retrieve_chunks(question)
    context="\n\n".join(doc.page_content for doc in docs)
    prompt,llm=get_prompt_and_llm()
    final_prompt=prompt.format(context=context,question=question)
    response=llm.invoke(final_prompt)
    return response.content

In [110]:
answer_question("what is multi head attention")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1485.71it/s]


[{'type': 'text',
  'text': '*   Multi-head attention is a mechanism that allows a model to look at different parts of information at the same time from various "representation subspaces." \n*   Instead of doing just one single attention calculation, the model runs several attention functions in parallel (using "heads").\n*   This approach is beneficial because using only a single attention head can cause the model to average out information, which limits its effectiveness. \n*   By using multiple heads, the model can jointly process different types of information, improving its performance and overall quality.\n*   The model uses these multiple heads by linearly projecting the original data into different, learned versions, and the final result is calculated by combining these individual outputs.',
  'extras': {'signature': 'EnAKbgERTTIP9XLgc6cy8TiU21UKyLQI0zPzWiRI3rvmYfZcaFCOSL2ohQlv1bElHYxTpLIp4xOSD81a0Uh7fqIZccb8ntduIon4sK7A7Vt/nLqVy/iLkrKJ1xU8b4GmJPE86fqESL8KYuwyJ7kD4oHw'}}]